# 2D Al FCC Layer with Alpha Particle Trajectory

This notebook builds a single-layer FCC Al lattice, places an alpha particle next to it, and renders a wiggly path with an arrow showing the alpha-particle direction.

In [ ]:
import numpy as np
import k3d


def build_wiggly_path(start, end, amplitude=0.55, wiggles=3, num_points=350):
    t = np.linspace(0.0, 1.0, num_points, dtype=np.float32)
    direction = end - start

    perp = np.array([-direction[1], direction[0], 0.0], dtype=np.float32)
    norm = np.linalg.norm(perp)
    if norm < 1e-8:
        perp = np.array([0.0, 1.0, 0.0], dtype=np.float32)
    else:
        perp = perp / norm

    centerline = start[None, :] + t[:, None] * direction[None, :]
    offsets = (amplitude * np.sin(2.0 * np.pi * wiggles * t)).astype(np.float32)
    return (centerline + offsets[:, None] * perp[None, :]).astype(np.float32)


def build_alpha_cluster(center, spacing=0.75):
    # Compact overlap style in xy plane.
    offsets = np.array(
        [
            [0.0, 0.5 * spacing, 0.0],
            [0.0, -0.5 * spacing, 0.0],
            [-0.5 * spacing, 0.0, 0.0],
            [0.5 * spacing, 0.0, 0.0],
        ],
        dtype=np.float32,
    )

    # Rotate whole alpha-cluster by +90 degrees around x-axis.
    theta = np.pi / 2.0
    rot_x = np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, np.cos(theta), -np.sin(theta)],
            [0.0, np.sin(theta), np.cos(theta)],
        ],
        dtype=np.float32,
    )
    offsets = offsets @ rot_x.T

    nucleon_positions = center[None, :] + offsets
    nucleon_colors = np.array([0xFF3B30, 0xFF3B30, 0x2E6BFF, 0x2E6BFF], dtype=np.uint32)
    return nucleon_positions.astype(np.float32), nucleon_colors


def build_al_fcc_layer_scene(nx=10, ny=4, lattice_constant=4.05):
    basis = np.array(
        [
            [0.0, 0.0, 0.0],
            [0.5 * lattice_constant, 0.5 * lattice_constant, 0.0],
        ],
        dtype=np.float32,
    )

    points = []
    for i in range(nx):
        for j in range(ny):
            origin = np.array([i * lattice_constant, j * lattice_constant, 0.0], dtype=np.float32)
            points.append(origin + basis[0])
            points.append(origin + basis[1])
    lattice_points = np.array(points, dtype=np.float32)

    xmin, ymin = lattice_points[:, 0].min(), lattice_points[:, 1].min()
    xmax, ymax = lattice_points[:, 0].max(), lattice_points[:, 1].max()
    ymid = 0.5 * (ymin + ymax)

    alpha_position = np.array([xmin - 12.0, ymid, 0.0], dtype=np.float32)
    alpha_cluster_positions, alpha_cluster_colors = build_alpha_cluster(alpha_position, spacing=0.75)

    path_end = np.array([xmax + 1.0, ymid, 0.0], dtype=np.float32)
    path = build_wiggly_path(alpha_position, path_end, amplitude=0.55, wiggles=7, num_points=350)

    # Built-in arrow attached at the line end, kept in-plane.
    tail_idx = max(0, len(path) - 12)
    tangent = path[-1] - path[tail_idx]
    tangent_norm = np.linalg.norm(tangent)
    if tangent_norm < 1e-8:
        tangent = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        tangent_norm = 1.0
    tangent = tangent / tangent_norm

    arrow_origin = path[-1]
    arrow_vector = 2.6 * tangent

    # Keep the blue sheet slightly below z=0 so line/arrow remain visible on the plane.
    sheet_margin = 0.5
    sheet_z = -0.08
    sheet_vertices = np.array(
        [
            [xmin - sheet_margin, ymin - sheet_margin, sheet_z],
            [xmax + sheet_margin, ymin - sheet_margin, sheet_z],
            [xmax + sheet_margin, ymax + sheet_margin, sheet_z],
            [xmin - sheet_margin, ymax + sheet_margin, sheet_z],
        ],
        dtype=np.float32,
    )
    sheet_indices = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.uint32)

    plot = k3d.plot(grid_visible=False, axes_helper=1.0)
    plot.background_color = 0x040814
    plot.camera = [
        float(0.1 * (xmin + xmax)),
        float(0.5 * (ymin + ymax)),
        80.0,
        float(0.5 * (xmin + xmax)),
        float(0.5 * (ymin + ymax)),
        0.0,
        0.0,
        1.0,
        0.0,
    ]

    plot += k3d.mesh(
        vertices=sheet_vertices,
        indices=sheet_indices,
        color=0x1E90FF,
        opacity=0.2,
        name="lattice sheet",
    )

    plot += k3d.points(
        positions=lattice_points,
        point_size=0.8,
        color=0xA9A9A9,
        shader="3d",
        name="Al FCC (001) single layer",
    )

    plot += k3d.points(
        positions=alpha_cluster_positions,
        point_size=1.35,
        colors=alpha_cluster_colors,
        shader="3d",
        name="alpha particle (2p + 2n)",
    )

    plot += k3d.line(vertices=path, width=0.28, color=0x00BFFF, name="alpha trajectory")

    plot += k3d.vectors(
        origins=np.array([arrow_origin], dtype=np.float32),
        vectors=np.array([arrow_vector], dtype=np.float32),
        colors=[0xFFD60A],
        line_width=0.1,
        head_size=2.0,
        name="trajectory arrow",
    )

    return plot


scene = build_al_fcc_layer_scene()
scene

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…